# Dev Vault - Qwen3.8-27B en Kaggle (llama.cpp + web UI + tunel)

Levanta **Qwen3.8-27B** (con vision) en la GPU de Kaggle con `llama-server`. Si algo falla, prueba solo las siguientes
opciones: Qwen3.8 con menos contexto -> Qwen3.6-27B -> Qwen3.6 con menos contexto.

Por la misma URL del tunel quedan disponibles:
- **Web UI** de llama.cpp (te pide tu API key)
- **API compatible con OpenAI** en `/v1/chat/completions`

**Tuneles:** abre a la vez todos los que tengan credenciales (ngrok, Tailscale Funnel, Cloudflare) y el cliente cambia solo al que funcione si uno falla o se queda sin cupo.

**Dos formas de usarlo:**
- **Manual** (Run All desde la web de Kaggle): crea los Secrets `API_KEY` y, si quieres URL fija, `NGROK_AUTHTOKEN` y `NGROK_DOMAIN`.
- **Desde tu terminal** con `client/vault.py`: la configuracion viaja en un Dataset privado `vault-config` (ver instructions/client-guide.md).

**Ajustes de la sesion:** Accelerator **GPU T4 x2**, Internet **On**, Visibility **Private**.

El notebook se apaga solo si no hay uso durante `IDLE_MINUTES`, al pasar `MAX_HOURS`, o cuando el cliente envia la senal `vault down`.

In [ ]:
# ===== 1. CONFIGURACION =====
import os, re, sys, json, time, glob, shutil, secrets, hmac, hashlib, threading, collections, subprocess
import urllib.request, urllib.error

# Cadena de intentos: (nombre, [repos GGUF a probar en orden], contexto)
Q38 = ["ggml-org/Qwen3.8-27B-GGUF", "unsloth/Qwen3.8-27B-GGUF", "bartowski/Qwen_Qwen3.8-27B-GGUF"]
Q36 = ["ggml-org/Qwen3.6-27B-GGUF", "unsloth/Qwen3.6-27B-GGUF", "bartowski/Qwen_Qwen3.6-27B-GGUF"]
# PRUEBA DE CONTEXTO: LONG_MODE=True prueba 196608 (riesgo de OOM); por defecto empieza en 131072.
LONG_MODE = False   # True = prueba primero 196608 (menos de 1 GB de VRAM libre por GPU; riesgo de quedarse sin memoria)
ATTEMPTS = ([("Qwen3.8-27B", Q38, 196608)] if LONG_MODE else []) + [
    ("Qwen3.8-27B", Q38, 131072),
    ("Qwen3.8-27B", Q38, 98304),
    ("Qwen3.8-27B", Q38, 65536),
    ("Qwen3.8-27B", Q38, 32768),
    ("Qwen3.6-27B", Q36, 65536),
    ("Qwen3.6-27B", Q36, 32768),
]
KV_TYPE = "q8_0"        # cache de contexto comprimida ("f16" = sin comprimir, "q4_0" = mas pequena)
KV_ON_CPU = False       # True = la cache de contexto va a la RAM (-nkvo): cabe mas, pero es mas lento
QUANTS = ["Q4_K_M", "Q4_K_S", "IQ4_XS", "Q4_0", "Q3_K_M"]   # orden de preferencia
THINKING = False        # False = mas rapido (recomendado en T4). True = razona antes de responder
PORT = 8080
TUNNEL = "auto"         # "auto" = todos los que tengan credenciales | o una lista: "ngrok,tailscale,cloudflared"
LLAMA_REF = "master"    # rama/tag de llama.cpp a compilar
ALLOW_COMMUNITY_BINARY = False  # True = si no hay nvcc, descarga un binario de TERCEROS (solo T4). Ver riesgos.
IDLE_MINUTES = 30       # apaga si no hay uso durante este tiempo (0 = nunca)
MAX_HOURS = 8           # tope de seguridad de la sesion (0 = sin tope)
BUILD_JOBS = os.cpu_count() or 4
START = time.time()

def phase(msg):
    print(f"[VAULT] {msg}", flush=True)

def _load_cfg_file():
    for p in glob.glob("/kaggle/input/**/vault-config.json", recursive=True):
        try:
            return json.load(open(p, encoding="utf-8"))
        except Exception:
            pass
    return {}
CFG_FILE = _load_cfg_file()

def get_secret(name):
    """Orden: Kaggle Secrets -> variable de entorno -> Dataset privado vault-config."""
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name) or CFG_FILE.get(name)

API_KEY = get_secret("API_KEY")
if not API_KEY:
    API_KEY = secrets.token_urlsafe(32)
    print("AVISO: no hay API_KEY (ni Secret ni Dataset). Clave temporal de esta sesion:", API_KEY)
NGROK_TOKEN = get_secret("NGROK_AUTHTOKEN")
NGROK_DOMAIN = (get_secret("NGROK_DOMAIN") or "").replace("https://", "").replace("http://", "").strip("/ ")
NTFY_TOPIC = get_secret("NTFY_TOPIC")
TUNNEL = get_secret("TUNNEL") or TUNNEL
TS_AUTHKEY = get_secret("TS_AUTHKEY")
TS_HOSTNAME = get_secret("TS_HOSTNAME") or "kaggle-ia"
TS_MODE = (get_secret("TS_MODE") or "funnel").lower()   # funnel = publico (protegido por API key) | serve = solo tu tailnet
_cf = get_secret("CLOUDFLARED_ALSO")
CLOUDFLARED_ALSO = str(True if _cf is None else _cf).lower() in ("1", "true", "yes")
IDLE_MINUTES = int(get_secret("IDLE_MINUTES") or IDLE_MINUTES)
MAX_HOURS = float(get_secret("MAX_HOURS") or MAX_HOURS)
print("Config OK | cores:", BUILD_JOBS, "| tuneles:", TUNNEL, "| ngrok:", bool(NGROK_TOKEN), "| tailscale:", bool(TS_AUTHKEY),
      "| apagado remoto:", bool(NTFY_TOPIC), "| inactividad:", IDLE_MINUTES, "min | tope:", MAX_HOURS, "h")

In [ ]:
# ===== 2. DIAGNOSTICO DEL ENTORNO =====
phase("fase=diagnostico")
def sh(cmd, check=False, timeout=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if check and r.returncode != 0:
        raise RuntimeError(f"{cmd}\n{r.stdout}\n{r.stderr}")
    return r

print(sh("nvidia-smi --query-gpu=name,memory.total --format=csv").stdout)
print(sh("free -g | head -2").stdout)
print(sh("df -h /kaggle/working /tmp /kaggle/temp 2>/dev/null").stdout)

if not sh("nvidia-smi -L").stdout.strip():
    raise SystemExit("No hay GPU. Settings -> Accelerator -> GPU T4 x2 y reinicia la sesion.")
caps = [l.strip() for l in sh("nvidia-smi --query-gpu=compute_cap --format=csv,noheader").stdout.splitlines()
        if re.fullmatch(r"\d+\.\d+", l.strip())]
CUDA_ARCHS = sorted({c.replace(".", "") for c in caps}) or ["60", "75"]   # P100=60, T4=75
print("Arquitecturas CUDA:", CUDA_ARCHS)

# nvcc: lo necesita la compilacion (no hace falta si usas un binario guardado en un Dataset)
NVCC = shutil.which("nvcc") or next(iter(sorted(glob.glob("/usr/local/cuda*/bin/nvcc"))), None)
print("nvcc:", NVCC or "NO ENCONTRADO")

try:
    urllib.request.urlopen("https://huggingface.co", timeout=10)
    print("Internet OK")
except Exception as e:
    raise SystemExit(f"Sin internet ({e}). Settings -> Internet: On (requiere telefono verificado).")

In [ ]:
# ===== 3. llama-server: reutilizar binario guardado o compilar =====
phase("fase=binario")
WORK = "/kaggle/working"
BIN_DIR = f"{WORK}/llama-bin"

def find_prebuilt():
    for p in glob.glob("/kaggle/input/**/llama-server", recursive=True):
        return os.path.dirname(p)
    return None

def cuda_driver_stub():
    """En Kaggle suele faltar libcuda.so (sin version) y cmake no define CUDA::cuda_driver."""
    if glob.glob("/usr/local/cuda/lib64/stubs/libcuda.so"):
        return True
    for real in ("/usr/lib/x86_64-linux-gnu/libcuda.so.1", "/usr/lib64/libcuda.so.1",
                 "/usr/local/cuda/compat/libcuda.so.1"):
        if os.path.exists(real):
            os.makedirs("/usr/local/cuda/lib64/stubs", exist_ok=True)
            sh(f"ln -sf {real} /usr/local/cuda/lib64/stubs/libcuda.so")
            print("stub del driver ->", real)
            return True
    return False

def build_from_source():
    os.environ["PATH"] = os.path.dirname(NVCC) + ":" + os.environ["PATH"]
    os.environ["CUDACXX"] = NVCC
    sh("apt-get update -qq && apt-get install -y -qq cmake build-essential git > /dev/null", check=False)
    SRC = "/tmp/llama.cpp"
    if not os.path.isdir(SRC):
        sh(f"git clone --depth 1 --branch {LLAMA_REF} https://github.com/ggml-org/llama.cpp {SRC}", check=True)
    archs = ";".join(CUDA_ARCHS)
    base = (f"cmake -S {SRC} -B {SRC}/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES='{archs}' "
            f"-DCMAKE_BUILD_TYPE=Release -DLLAMA_OPENSSL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF")
    cuda_driver_stub()
    r = None
    for i, cfg in enumerate([base, base + " -DGGML_CUDA_NO_VMM=ON"]):
        if i:
            print("Reintento del configure con -DGGML_CUDA_NO_VMM=ON")
            sh(f"rm -rf {SRC}/build")
        r = sh(cfg)
        print(r.stdout[-1500:], r.stderr[-1500:])
        if r.returncode == 0:
            break
    if r.returncode != 0:
        raise SystemExit("Fallo cmake configure. Revisa el mensaje de arriba.")
    phase("fase=compilando (~25 min)")
    t0 = time.time()
    r = sh(f"cmake --build {SRC}/build --target llama-server -j {BUILD_JOBS} 2>&1 | tail -40")
    print(r.stdout)
    if not os.path.exists(f"{SRC}/build/bin/llama-server"):
        raise SystemExit("La compilacion no produjo llama-server. Revisa la salida de arriba.")
    os.makedirs(BIN_DIR, exist_ok=True)
    for f in glob.glob(f"{SRC}/build/bin/*"):
        if os.path.isfile(f) and (f.endswith(".so") or ".so." in f or os.path.basename(f) == "llama-server"):
            shutil.copy2(f, BIN_DIR)
    print(f"Compilado en {(time.time()-t0)/60:.1f} min")

def download_community_binary():
    # Binario de TERCEROS (no auditado) compilado solo para T4 (sm_75). Usalo bajo tu propio riesgo.
    sh("pip install -q -U huggingface_hub", check=False)
    from huggingface_hub import list_repo_files, hf_hub_download
    repo = "mboy2k/llama-cpp-sm75-cuda-binary"
    print("AVISO: descargando binario de terceros:", repo)
    os.makedirs(BIN_DIR, exist_ok=True)
    for f in list_repo_files(repo):
        if f == "llama-server" or ".so" in f:
            hf_hub_download(repo, f, local_dir=BIN_DIR)

pre = find_prebuilt()
if pre:
    print("Usando binario guardado en Dataset:", pre)
    if not os.path.isdir(BIN_DIR):
        shutil.copytree(pre, BIN_DIR)
elif NVCC:
    print("No hay binario guardado. Compilando llama.cpp (tarda ~25 min)...")
    build_from_source()
elif ALLOW_COMMUNITY_BINARY and "75" in CUDA_ARCHS:
    download_community_binary()
else:
    raise SystemExit(
        "No hay nvcc (compilador CUDA) ni binario guardado.\n"
        "Opciones: (a) compila en otro entorno con CUDA y sube la carpeta llama-bin como Dataset, o\n"
        "(b) pon ALLOW_COMMUNITY_BINARY=True en la celda 1 (binario de terceros solo T4; lee los riesgos en el README).")
os.chmod(f"{BIN_DIR}/llama-server", 0o755)

LLAMA_SERVER = f"{BIN_DIR}/llama-server"
ENV = dict(os.environ, LD_LIBRARY_PATH=f"{BIN_DIR}:" + os.environ.get("LD_LIBRARY_PATH", ""))
v = subprocess.run([LLAMA_SERVER, "--version"], env=ENV, capture_output=True, text=True)
print((v.stdout + v.stderr).strip()[-400:] or "sin salida de version")
if v.returncode != 0:
    raise SystemExit("llama-server no arranca (faltan librerias?). Revisa la salida de arriba.")

In [ ]:
# ===== 4. DESCARGA DEL MODELO (elige carpeta con espacio, cuantizacion y mmproj automaticamente) =====
phase("fase=descargando modelo")
sh("pip install -q -U huggingface_hub", check=False)
from huggingface_hub import HfApi, hf_hub_download

def free_gb(path):
    try:
        return shutil.disk_usage(path).free / 1e9
    except Exception:
        return 0

def pick_dir(need_gb):
    cands = [d for d in ["/kaggle/temp", "/tmp", "/kaggle/working"] if os.path.isdir(d)]
    cands.sort(key=free_gb, reverse=True)
    for d in cands:
        print(f"  espacio libre en {d}: {free_gb(d):.1f} GB")
    best = cands[0]
    return best if free_gb(best) >= need_gb + 2 else None

def resolve_files(repo):
    """Devuelve (lista_de_shards_gguf, nombre_primer_shard, mmproj_o_None, tamano_GB, cuantizacion) o None."""
    info = HfApi().model_info(repo, files_metadata=True)
    files = {s.rfilename: (s.size or 0) for s in info.siblings}
    ggufs = [f for f in files if f.lower().endswith(".gguf")
             and not re.search(r"(^|/)(mmproj|mtp|dflash)", f.lower())]
    for q in QUANTS:
        pat = re.compile(re.escape(q), re.I)
        match = sorted(f for f in ggufs if pat.search(f))
        if match:
            size = sum(files[f] for f in match) / 1e9
            mm = [f for f in files if "mmproj" in f.lower() and f.lower().endswith(".gguf")]
            mm.sort(key=lambda f: (0 if re.search(r"q8_0", f, re.I) else 1 if re.search(r"f16", f, re.I) else 2, f))
            return match, match[0], (mm[0] if mm else None), size, q
    return None

MODEL_CACHE = {}
def fetch_model(name, repos):
    if name in MODEL_CACHE:
        return MODEL_CACHE[name]
    for repo in repos:
        try:
            res = resolve_files(repo)
        except Exception as e:
            print(f"  [{repo}] no disponible: {str(e)[:120]}")
            continue
        if not res:
            print(f"  [{repo}] sin cuantizaciones utiles")
            continue
        shards, first, mm, size, q = res
        need = size + (0.9 if mm else 0)
        d = pick_dir(need)
        if d is None:
            print(f"  [{repo}] {q} pesa {size:.1f} GB y no hay espacio suficiente")
            continue
        dest = f"{d}/models/{name}"
        os.makedirs(dest, exist_ok=True)
        print(f"  Descargando {repo} ({q}, {size:.1f} GB) -> {dest}")
        try:
            for f in shards:
                hf_hub_download(repo, f, local_dir=dest)
            mm_path = hf_hub_download(repo, mm, local_dir=dest) if mm else None
        except Exception as e:
            print(f"  [{repo}] fallo la descarga: {str(e)[:160]}")
            continue
        if not mm_path:
            print("  AVISO: este repo no trae mmproj -> el modelo funcionara SIN vision")
        out = (f"{dest}/{first}", mm_path, repo, q)
        MODEL_CACHE[name] = out
        return out
    return None
print("Funciones de descarga listas")

In [ ]:
# ===== 5. ARRANQUE CON RESPALDO AUTOMATICO =====
LOG = f"{WORK}/llama-server.log"
server = None

def stop_server():
    global server
    if server and server.poll() is None:
        server.terminate()
        try: server.wait(timeout=20)
        except Exception: server.kill()
    server = None

def start_server(model_path, mmproj, ctx, alias):
    global server
    sampling = ("--temp 1.0 --top-p 0.95 --top-k 20" if THINKING
                else "--temp 0.7 --top-p 0.8 --top-k 20 --presence-penalty 1.5")
    kwargs = json.dumps({"enable_thinking": bool(THINKING)})
    cmd = [LLAMA_SERVER, "-m", model_path, "--alias", alias, "--host", "127.0.0.1", "--port", str(PORT),
           "-ngl", "99", "-c", str(ctx), "--jinja", "--metrics", "--api-key", API_KEY,
           "-ctk", KV_TYPE, "-ctv", KV_TYPE, "-fa", "on", "-np", "1",
           "--chat-template-kwargs", kwargs] + sampling.split()
    if KV_ON_CPU:
        cmd += ["-nkvo"]
    if mmproj:
        cmd += ["--mmproj", mmproj]
    print("Lanzando:", " ".join(c if c != API_KEY else "****" for c in cmd))
    server = subprocess.Popen(cmd, env=ENV, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

def wait_ready(timeout=900):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if server.poll() is not None:
            return False
        try:
            if urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=3).status == 200:
                return True
        except Exception:
            pass
        time.sleep(5)
    return False

ACTIVE = None
for name, repos, ctx in ATTEMPTS:
    print(f"\n=== Intento: {name} | contexto {ctx} ===")
    phase(f"fase=cargando {name} ctx={ctx}")
    got = fetch_model(name, repos)
    if not got:
        print("  No se pudo obtener este modelo, paso al siguiente.")
        continue
    path, mm, repo, q = got
    start_server(path, mm, ctx, name)
    if wait_ready():
        ACTIVE = dict(name=name, repo=repo, quant=q, ctx=ctx, vision=bool(mm))
        print("LISTO ->", ACTIVE)
        phase(f"fase=vram ctx={ctx} kv={KV_TYPE}")
        print(sh("nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv").stdout)
        try:
            body = json.dumps({
                "model": name,
                "messages": [{"role": "user", "content": "Responde solo: ok"}],
                "max_tokens": 16,
                "temperature": 0,
                "stream": False,
            }).encode("utf-8")
            req = urllib.request.Request(
                f"http://127.0.0.1:{PORT}/v1/chat/completions",
                data=body,
                headers={"Content-Type": "application/json",
                         "Authorization": f"Bearer {API_KEY}"},
                method="POST",
            )
            with urllib.request.urlopen(req, timeout=300) as resp:
                tim = json.loads(resp.read().decode("utf-8")).get("timings") or {}
            pp = tim.get("prompt_per_second")
            tp = tim.get("predicted_per_second")
            if pp is not None and tp is not None:
                print(f"Velocidad: lectura {pp:.2f} tok/s | escritura {tp:.2f} tok/s")
            else:
                print("timings no disponible")
        except Exception:
            print("timings no disponible")
        break
    print("  Fallo al arrancar. Ultimas lineas del log:")
    print(sh(f"tail -25 {LOG}").stdout)
    stop_server()

if not ACTIVE:
    phase("fase=error: ningun modelo arranco")
    raise SystemExit("Ningun intento funciono. Revisa llama-server.log (junto a este notebook).")

In [ ]:
# ===== 6. PRUEBA RAPIDA (texto + vision) =====
import base64, io
def chat(content, max_tokens=64):
    body = json.dumps({"model": ACTIVE["name"], "messages": [{"role": "user", "content": content}],
                       "max_tokens": max_tokens, "temperature": 0.2}).encode()
    req = urllib.request.Request(f"http://127.0.0.1:{PORT}/v1/chat/completions", data=body,
                                 headers={"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"})
    r = json.loads(urllib.request.urlopen(req, timeout=300).read())
    m = r["choices"][0]["message"]
    return (m.get("content") or "").strip() or ("[solo razonamiento] " + (m.get("reasoning_content") or "")[:200])

t0 = time.time()
print("Texto :", chat("Escribe en Python una funcion que invierta una cadena. Solo el codigo.", 120))
print(f"({time.time()-t0:.1f}s)")

if ACTIVE["vision"]:
    try:
        from PIL import Image
        im = Image.new("RGB", (128, 128), (220, 20, 20))
        buf = io.BytesIO(); im.save(buf, "PNG")
        uri = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()
        print("Vision:", chat([{"type": "text", "text": "De que color es esta imagen? Una palabra."},
                               {"type": "image_url", "image_url": {"url": uri}}], 16))
    except Exception as e:
        print("AVISO prueba de vision fallo:", str(e)[:200])

In [ ]:
# ===== 7. TUNELES EN PARALELO: ngrok + Tailscale Funnel + Cloudflare (el cliente cambia solo al que funcione) =====
phase("fase=abriendo tuneles")

def _drain(proc, buf):
    for line in proc.stdout:
        buf.append(line.rstrip())

def ensure_ngrok():
    sh("pip install -q pyngrok", check=False)
    from pyngrok import ngrok as _n, conf as _c
    _n.install_ngrok()
    return _c.get_default().ngrok_path

def start_ngrok():
    path = ensure_ngrok()
    env = dict(os.environ, NGROK_AUTHTOKEN=NGROK_TOKEN)
    variants = ([f"--url=https://{NGROK_DOMAIN}"], [f"--domain={NGROK_DOMAIN}"]) if NGROK_DOMAIN else ([],)
    for extra in variants:
        cmd = [path, "http", str(PORT)] + extra + ["--log=stdout", "--log-format=json"]
        p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        buf = collections.deque(maxlen=300)
        threading.Thread(target=_drain, args=(p, buf), daemon=True).start()
        for _ in range(45):
            time.sleep(1)
            if p.poll() is not None:
                break
            text = "\n".join(buf)
            if "started tunnel" in text or ("client session established" in text and "url=" in text):
                m = re.search(r'"url":\s*"(https://[^"]+)"', text) or re.search(r"url=(https://\S+)", text)
                url = f"https://{NGROK_DOMAIN}" if NGROK_DOMAIN else (m.group(1) if m else None)
                if url:
                    return [p], url
        print("  ngrok no arranco con", extra or "dominio aleatorio", "- ultimas lineas:")
        print("\n".join(list(buf)[-8:]))
        if p.poll() is None:
            p.terminate()
    return None, None

def start_tailscale():
    # Tailscale en modo userspace (no necesita /dev/net/tun) + Funnel (publico) o Serve (solo tu tailnet)
    if not shutil.which("tailscaled"):
        r = sh("curl -fsSL https://tailscale.com/install.sh | sh", timeout=900)
        print((r.stdout + r.stderr)[-500:])
    if not shutil.which("tailscaled"):
        print("  tailscale no se pudo instalar")
        return None, None
    sock, sdir = "/tmp/tailscaled.sock", "/tmp/tailscale"
    os.makedirs(sdir, exist_ok=True)
    d = subprocess.Popen(["tailscaled", "--tun=userspace-networking", f"--statedir={sdir}", f"--socket={sock}"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    dbuf = collections.deque(maxlen=200)
    threading.Thread(target=_drain, args=(d, dbuf), daemon=True).start()
    for _ in range(30):
        if os.path.exists(sock):
            break
        time.sleep(1)
    up = subprocess.run(["tailscale", f"--socket={sock}", "up", f"--auth-key={TS_AUTHKEY}",
                         f"--hostname={TS_HOSTNAME}", "--accept-dns=false"], capture_output=True, text=True, timeout=180)
    if up.returncode != 0:
        print("  tailscale up fallo:", (up.stdout + up.stderr)[-400:].replace(TS_AUTHKEY, "****"))
        d.terminate()
        return None, None
    mode = "serve" if TS_MODE == "serve" else "funnel"
    p = subprocess.Popen(["tailscale", f"--socket={sock}", mode, "--yes", str(PORT)],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    buf = collections.deque(maxlen=200)
    threading.Thread(target=_drain, args=(p, buf), daemon=True).start()
    for _ in range(60):
        time.sleep(1)
        m = re.search(r"https://[A-Za-z0-9.-]+\.ts\.net", "\n".join(buf))
        if m:
            return [d, p], m.group(0)
        if p.poll() is not None:
            break
    print(f"  tailscale {mode} no dio URL. Salida (puede traer un enlace para habilitar Funnel una sola vez):")
    print("\n".join(list(buf)[-10:]))
    for x in (p, d):
        if x.poll() is None:
            x.terminate()
    return None, None

def start_cloudflared():
    if not os.path.exists("/usr/local/bin/cloudflared"):
        sh("curl -fsSL -o /usr/local/bin/cloudflared "
           "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
           "&& chmod +x /usr/local/bin/cloudflared", check=True)
    p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    buf = collections.deque(maxlen=300)
    threading.Thread(target=_drain, args=(p, buf), daemon=True).start()
    for _ in range(60):
        time.sleep(1)
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", "\n".join(buf))
        if m:
            return [p], m.group(0)
        if p.poll() is not None:
            break
    return None, None

ORDER = ["ngrok", "tailscale", "cloudflared"]   # prioridad del URL principal (los dos primeros soportan streaming)
WANT = ORDER if TUNNEL in ("auto", "", None) else [t.strip() for t in TUNNEL.split(",") if t.strip()]
TUN = {}    # nombre -> {"procs": [...], "url": "https://..."}

def try_start(name, fn, enabled, why):
    if name not in WANT:
        return
    if not enabled:
        print(f"  [{name}] omitido: {why}")
        return
    try:
        procs, url = fn()
    except Exception as e:
        print(f"  [{name}] error: {str(e)[:200]}")
        return
    if url:
        TUN[name] = {"procs": procs, "url": url}
        print(f"  [{name}] OK -> {url}")
    else:
        print(f"  [{name}] no se pudo abrir")

try_start("ngrok", start_ngrok, bool(NGROK_TOKEN), "falta NGROK_AUTHTOKEN")
try_start("tailscale", start_tailscale, bool(TS_AUTHKEY), "falta TS_AUTHKEY")
try_start("cloudflared", start_cloudflared, CLOUDFLARED_ALSO or not TUN, "CLOUDFLARED_ALSO=false")

if not TUN:
    phase("fase=error: no se pudo abrir ningun tunel")
    raise SystemExit("No se pudo abrir ningun tunel. Revisa NGROK_AUTHTOKEN / TS_AUTHKEY o la conexion a Internet.")

PRIMARY = next(n for n in ORDER if n in TUN)
PUBLIC_URL = TUN[PRIMARY]["url"]
if PRIMARY == "cloudflared":
    print("AVISO: solo hay Cloudflare Quick Tunnel, que NO soporta streaming SSE (lo dice su documentacion): "
          "la web UI puede fallar. El cliente vault.py usa respuestas sin streaming con este tunel.")

def print_markers():
    print(f"VAULT_MODEL={ACTIVE['name']}", flush=True)
    for n, t in TUN.items():
        print(f"VAULT_URL_{n.upper()}={t['url']}", flush=True)
    print(f"VAULT_URL={PUBLIC_URL}", flush=True)

print("=" * 60)
print("Modelo     :", ACTIVE["name"], f"({ACTIVE['quant']}, ctx {ACTIVE['ctx']}, vision={ACTIVE['vision']})")
for n, t in TUN.items():
    print(f"Tunel {n:<11}:", t["url"], "(principal)" if n == PRIMARY else "")
print("Web UI     :", PUBLIC_URL, " (pega tu API_KEY cuando la pida)")
print("API OpenAI :", PUBLIC_URL + "/v1")
print("=" * 60)
print_markers()
phase("fase=listo")

In [ ]:
# ===== 8. MANTENER VIVA LA SESION + APAGADO (remoto, por inactividad o por tope de horas) =====
STOP = threading.Event()
STOP_REASON = ""
LAST_ACTIVITY = time.time()

def stop_signature():
    return hmac.new(API_KEY.encode(), b"vault-stop", hashlib.sha256).hexdigest()

def ntfy_listener():
    # Escucha la senal 'vault down' del cliente (ntfy.sh, gratis y sin cuenta)
    global STOP_REASON
    while not STOP.is_set():
        try:
            url = f"https://ntfy.sh/{NTFY_TOPIC}/json?since={int(START)}"
            with urllib.request.urlopen(url, timeout=120) as r:
                for line in r:
                    if STOP.is_set():
                        return
                    try:
                        ev = json.loads(line)
                    except Exception:
                        continue
                    if ev.get("event") == "message" and hmac.compare_digest(
                            (ev.get("message") or "").strip(), "STOP " + stop_signature()):
                        STOP_REASON = "senal de apagado recibida (vault down)"
                        STOP.set()
                        return
        except Exception:
            time.sleep(5)

def read_activity():
    req = urllib.request.Request(f"http://127.0.0.1:{PORT}/metrics", headers={"Authorization": f"Bearer {API_KEY}"})
    txt = urllib.request.urlopen(req, timeout=5).read().decode()
    vals = {m.group(1): float(m.group(2)) for m in re.finditer(r"^llamacpp:(\w+)\s+([0-9.eE+-]+)\s*$", txt, re.M)}
    return (vals.get("prompt_tokens_total", 0) + vals.get("tokens_predicted_total", 0),
            vals.get("requests_processing", 0))

def tunnels_alive():
    return {n for n, t in TUN.items() if all(p.poll() is None for p in t["procs"])}

if NTFY_TOPIC:
    threading.Thread(target=ntfy_listener, daemon=True).start()

last_total = 0
try:
    last_total, _ = read_activity()
except Exception:
    pass
LAST_ACTIVITY = time.time()
last_marker = 0
known_alive = tunnels_alive()

while not STOP.is_set():
    time.sleep(10)
    now = time.time()
    if now - last_marker > 300:
        print_markers()
        last_marker = now
    if server.poll() is not None:
        STOP_REASON = "llama-server se detuvo (revisa llama-server.log)"; break
    alive = tunnels_alive()
    for n in known_alive - alive:
        print(f"AVISO: el tunel {n} se detuvo", flush=True)
    known_alive = alive
    if not alive:
        STOP_REASON = "todos los tuneles se detuvieron"; break
    try:
        total, busy = read_activity()
        if total != last_total or busy > 0:
            last_total, LAST_ACTIVITY = total, now
    except Exception:
        pass
    if IDLE_MINUTES and now - LAST_ACTIVITY > IDLE_MINUTES * 60:
        STOP_REASON = f"sin uso durante {IDLE_MINUTES} min"; break
    if MAX_HOURS and now - START > MAX_HOURS * 3600:
        STOP_REASON = f"se alcanzo el tope de {MAX_HOURS} h"; break

STOP.set()
phase(f"fase=apagando: {STOP_REASON or 'detenido'}")
for t in TUN.values():
    for p in t["procs"]:
        if p.poll() is None:
            p.terminate()
stop_server()
phase("fase=apagado (la sesion termina ahora)")